# RAG Market Intelligence Pipeline

Ingest FRED press releases and tariff documents → embed with sentence-transformers →
store in ChromaDB → retrieve and answer market intelligence queries.

In [1]:
import sys
sys.path.insert(0, "..")

from core.rag_chain import ingest_documents, retrieve_chunks, get_rag_answer, build_rag_prompt
import json

## Step 1: Ingest Documents into ChromaDB

In [2]:
n_chunks = ingest_documents()
print(f"Total chunks ingested: {n_chunks}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Total chunks ingested: 7


## Step 2: Test Retrieval (Semantic Search)

In [3]:
query = "What was the market impact of Section 301 tariffs on EUR/USD?"
chunks = retrieve_chunks(query, top_k=3)

for i, chunk in enumerate(chunks):
    print(f"\n--- Chunk {i+1} (distance: {chunk['distance']}) ---")
    print(f"Source: {chunk['source']}")
    print(chunk['text'][:200] + "...")


--- Chunk 1 (distance: 0.2672) ---
Source: section_301_tariffs.txt
Section 301 Tariffs — Market Impact Analysis

On July 6, 2018, the United States imposed 25% tariffs on $34 billion worth of Chinese imports 
under Section 301 of the Trade Act of 1974. This marked th...

--- Chunk 2 (distance: 0.3189) ---
Source: section_301_tariffs.txt
billion of Chinese goods.
- May 10, 2019: Tariff rate increased to 25% on the same $200 billion tranche.
- Each escalation correlated with ADTV spikes in SPY and QQQ of 40-60% above trailing 20-day av...

--- Chunk 3 (distance: 0.5277) ---
Source: section_301_tariffs.txt
The S&P 500 fell 0.86% on the announcement day but recovered within 3 trading sessions.
- VIX spiked from 15.2 to 18.4 (21% increase) reflecting heightened uncertainty.
- GARCH(1,1) volatility persist...


## Step 3: Full RAG Pipeline

In [4]:
test_queries = [
    "What was the market impact of Section 301 tariffs on EUR/USD?",
    "What does the VIX indicate about current market conditions?",
    "When is the Fed expected to cut rates and how does it affect equities?",
    "What happened to GARCH volatility persistence after tariff announcements?",
]

for q in test_queries:
    print(f"\n{'='*60}")
    print(f"Q: {q}")
    print('='*60)
    result = get_rag_answer(q, top_k=2)
    print(f"\nAnswer:\n{result['answer'][:300]}...")
    print(f"\nSources: {[s['source'] for s in result['sources']]}")


Q: What was the market impact of Section 301 tariffs on EUR/USD?

Answer:
The market impact of the Section 301 tariffs on EUR/USD was a decline from 1.1690 to 1.1550 in the week following the initial announcement on July 6, 2018. This represents a 1.2% drop, driven by investors moving into USD safe-haven assets....

Sources: ['section_301_tariffs.txt', 'section_301_tariffs.txt']

Q: What does the VIX indicate about current market conditions?

Answer:
The VIX index data indicates that current market conditions are relatively calm but potentially unstable. Specifically:

- The average VIX in Q4 2023 was 14.2, which is well below the long-term average of 19.5, signaling low market volatility.
- However, this low VIX level is described as a "low VIX...

Sources: ['fred_release_2024.txt', 'section_301_tariffs.txt']

Q: When is the Fed expected to cut rates and how does it affect equities?

Answer:
Based on the provided context:

- The Federal Funds Rate has been held at 5.25-5.50% since Ju

## Step 4: View the Prompt Sent to LLM

In [5]:
chunks = retrieve_chunks("Section 301 tariff impact", top_k=3)
prompt = build_rag_prompt("What was the market impact of Section 301 tariffs on EUR/USD?", chunks)
print(prompt)

You are a quantitative market risk analyst. Answer the following question 
using ONLY the context provided below. If the context doesn't contain enough information, 
say so explicitly. Be precise and cite specific data points.

CONTEXT:
billion of Chinese goods.
- May 10, 2019: Tariff rate increased to 25% on the same $200 billion tranche.
- Each escalation correlated with ADTV spikes in SPY and QQQ of 40-60% above trailing 20-day average.

The Section 301 tariffs represent a structural shift in global trade policy that permanently 
altered correlation structures between FX pairs and equity indices.

---

Section 301 Tariffs — Market Impact Analysis

On July 6, 2018, the United States imposed 25% tariffs on $34 billion worth of Chinese imports 
under Section 301 of the Trade Act of 1974. This marked the beginning of the US-China trade war.

Market Impact:
- EUR/USD dropped from 1.1690 to 1.1550 in the week following the announcement, a 1.2% decline, 
  as investors fled to USD safe-hav

## Step 5: Test via FastAPI

With the server running (`uvicorn api.main:app --reload`), test:
```bash
curl -X POST http://localhost:8000/rag/query \
  -H "Content-Type: application/json" \
  -d '{"question": "What was the market impact of Section 301 tariffs on EUR/USD?"}'
```